# 08 — Full Backtest Analysis

**Strategy reference:** §20 (Testing Philosophy), §22 (V1 Boundaries).

Three overlay backtests we can run end-to-end on the existing
engines:

1. **Tide overlay** — `tide.tide_backtest.TideBacktester`
2. **Wave overlay** — `wave.wave_backtest.WaveBacktester` (isolated or stacked-on-Tide)
3. **Ripple** — `strategies.orderflow.backtest()` (C++ engine; requires the pybind11 module built)

We focus on Tide + Wave here because they only require OHLCV.
The Ripple cell will execute only if the C++ engine is available.

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

## 1. Tide overlay backtest
Tide doesn't trigger trades; its 'economic' performance is
measured by treating its bias × risk_multiplier as a positional
signal on close-to-close moves.

In [ ]:
from tide.tide_backtest import TideBacktester, TideStrategyParams, SizingMode

ohlcv = load_ohlcv('binance', 'BTCUSDT', '1m').tail(80_000).copy()
print(f'{len(ohlcv):,} bars  ({ohlcv.index.min()} → {ohlcv.index.max()})')

In [ ]:
params = TideStrategyParams(
    symbol='BTCUSDT', exchange='binance', timeframe='1m',
).with_timeframe('1m')
params

In [ ]:
bt = TideBacktester(params)
tide_result = bt.run(ohlcv)
print('keys :', list(getattr(tide_result, '__dict__', {}).keys()) or 'see below')
tide_result

In [ ]:
# Equity curve (TideBacktestResult stores it as a pandas Series by
# convention; we plot defensively in case the field name differs)
candidates = ['equity_curve', 'equity', 'pnl_series', 'nav']
eq = None
for name in candidates:
    if hasattr(tide_result, name):
        cand = getattr(tide_result, name)
        if isinstance(cand, pd.Series) and len(cand) > 0:
            eq = cand; break
if eq is not None:
    plot_equity_curve(eq, title=f'Tide overlay equity — {params.symbol}')
    plt.show()
else:
    print('No equity series field found on tide_result — inspect it manually above.')

## 2. Wave overlay backtest (isolated)
Wave by itself, TideBias pinned to NEUTRAL.  Permission-adjusted
sizing gives a clean economic readout of regime accuracy.

In [ ]:
from wave.wave_backtest import WaveBacktester, WaveStrategyParams

wparams = WaveStrategyParams(
    symbol='BTCUSDT', exchange='binance', timeframe='1m',
).with_timeframe('1m')
wbt = WaveBacktester(wparams)
wave_result = wbt.run(ohlcv)
wave_result

In [ ]:
eq = None
for name in candidates:
    if hasattr(wave_result, name):
        cand = getattr(wave_result, name)
        if isinstance(cand, pd.Series) and len(cand) > 0:
            eq = cand; break
if eq is not None:
    plot_equity_curve(eq, title='Wave overlay equity (isolated)')
    plt.show()

## 3. Stacked Wave-on-Tide
Feed Tide's per-bar bias into the Wave run.  This is the closest
approximation of how Wave will behave in production with a real
TideEngine upstream.

In [ ]:
wbt_stack = WaveBacktester(wparams)
try:
    wave_stacked = wbt_stack.run(ohlcv, tide_result=tide_result)
    print('stacked OK')
    print(wave_stacked)
except TypeError:
    print('WaveBacktester.run does not accept tide_result on this version; skipping')
    wave_stacked = None

## 4. Regime-conditional performance
Strip the per-bar regime label out of the Wave result and report
Sharpe per regime.

In [ ]:
from metrics import compute_sharpe_ratio

def per_regime_sharpe(result):
    if result is None:
        return None
    # Find a frame with a 'regime' column + a returns column
    for attr in ('bars', 'frame', 'features', 'history'):
        df = getattr(result, attr, None)
        if isinstance(df, pd.DataFrame) and 'regime' in df.columns:
            ret_col = next((c for c in ('log_return','return','ret') if c in df.columns), None)
            if ret_col:
                return df.groupby('regime')[ret_col].agg(
                    mean='mean', std='std',
                    sharpe=lambda s: compute_sharpe_ratio(list(s.dropna())),
                    count='count',
                )
    return None

per_regime_sharpe(wave_result)

## 5. Ripple backtest (C++ engine)
Requires the C++ orderflow module to be built (see
`backtestingCpp/orderflow/build.sh`).  The cell wraps the call in
try/except so the notebook stays runnable even when the build is
missing.

In [ ]:
try:
    from strategies.orderflow import backtest as ripple_backtest
    import time as _time

    params_ripple = {
        'tick_size': 0.01, 'imbalance_threshold': 3.0,
        'stacked_imbalance_levels': 3, 'absorption_volume_ratio': 5.0,
        'cvd_divergence_lookback': 100, 'exhaustion_lookback_bars': 5,
        'signal_strength_min': 0.3, 'paper_fills': True,
    }
    # Use the time range present in the local tick file
    ticks = load_ticks('BTCUSDT', max_trades=10)
    if len(ticks['trades']) == 0:
        raise RuntimeError('no trades in local tick file')
    from_ts = int(ticks['trades']['timestamp'].min())
    to_ts   = int(ticks['trades']['timestamp'].max())
    print(f'replaying ticks  {from_ts} → {to_ts}')
    t0 = _time.time()
    pnl, dd, ntrades, sharpe, cagr = ripple_backtest('binance', 'BTCUSDT',
                                                       from_ts, to_ts, params_ripple)
    print(f'pnl={pnl:.2f}  max_dd={dd:.2f}  trades={ntrades}  '
          f'sharpe={sharpe:.3f}  cagr={cagr:.3f}  took={_time.time()-t0:.1f}s')
except Exception as exc:
    print('Ripple backtest skipped:', type(exc).__name__, exc)

## Takeaways

* Tide and Wave overlays are cheap to iterate — they need only
  OHLCV and run in seconds.
* The Ripple backtest is the only one that needs the C++ engine
  built and a healthy tick HDF5; treat its results as the
  ground-truth strategy performance.
* Always check regime-conditional Sharpe before tuning thresholds:
  a strategy that is great in BREAKOUT and terrible in
  MEAN_REVERSION is a clue to tighten the permissions matrix.